In [6]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

# ==========================================
# 1. Automatic Environment Detection
# ==========================================
try:
    # --- Colab Environment ---
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
    print("☁️ Colab Environment Detected.")
except ImportError:
    # --- Local Environment (VS Code) ---
    from dotenv import load_dotenv
    current_dir = os.getcwd()
    project_root = os.path.abspath(os.path.join(current_dir, "..", ".."))
    env_path = os.path.join(project_root, ".env")
    load_dotenv(dotenv_path=env_path, override=True)
    print("💻 Local Environment Detected.")

# ==========================================
# 2. Initialize the Model
# ==========================================
model = ChatGoogleGenerativeAI(
    model="gemini-flash-lite-latest",
    temperature=0.1
)
print("🚀 Gemini model is ready!")

💻 Local Environment Detected.
🚀 Gemini model is ready!


In [7]:
# ==========================================
# Defining Independent Chains
# ==========================================

# 1. First Chain: The Poet
poem_prompt = ChatPromptTemplate.from_template("یک شعر دوبیتی کوتاه درباره {topic} بگو.")
poem_chain = poem_prompt | model | StrOutputParser()

# 2. Second Chain: The Storyteller
story_prompt = ChatPromptTemplate.from_template("یک داستان خیلی کوتاه درباره {topic} بنویس.")
story_chain = story_prompt | model | StrOutputParser()

# ==========================================
# Parallel Execution using RunnableParallel
# ==========================================

# RunnableParallel acts like a power strip, sending the input to all connected chains simultaneously
map_chain = RunnableParallel({
    "poem": poem_chain,
    "story": story_chain
})

# Execute the parallel chain
print("⏳ Executing chains in parallel. Please wait...\n")
result = map_chain.invoke({"topic": "پاییز"})

# Display the raw dictionary output
print("--- Raw Output Dictionary ---")
print(result)

⏳ Executing chains in parallel. Please wait...

--- Raw Output Dictionary ---
{'poem': 'برگ از درخت افتاد و زرد و خسته شد\nدشت از غم پاییز، چه دل\u200cشکسته شد\n\nابر آمد و بارید بر بامِ خانه\u200cها\nدروازه\u200cهای دل به روی غصه بسته شد', 'story': 'روی نیمکت چوبی پارک، پیرمرد چترش را مثل عصا دست گرفته بود و به رقص برگ\u200cهای زرد و نارنجی در هوا نگاه می\u200cکرد. \n\nباد خنکی وزید و یک برگ نارنجی\u200cرنگ، آرام روی شانه\u200cاش نشست. پیرمرد لبخندی زد، برگ را برداشت، عطر خاک باران\u200cخورده را نفس کشید و با خودش گفت: «پاییز، زیباترین بهانه\u200cی زمین برای دلتنگی است.»'}


In [8]:
# Accessing specific keys from the generated dictionary
print("🍂 شعر تولید شده:\n", result["poem"])
print("-" * 50)
print("🍁 داستان تولید شده:\n", result["story"])

🍂 شعر تولید شده:
 برگ از درخت افتاد و زرد و خسته شد
دشت از غم پاییز، چه دل‌شکسته شد

ابر آمد و بارید بر بامِ خانه‌ها
دروازه‌های دل به روی غصه بسته شد
--------------------------------------------------
🍁 داستان تولید شده:
 روی نیمکت چوبی پارک، پیرمرد چترش را مثل عصا دست گرفته بود و به رقص برگ‌های زرد و نارنجی در هوا نگاه می‌کرد. 

باد خنکی وزید و یک برگ نارنجی‌رنگ، آرام روی شانه‌اش نشست. پیرمرد لبخندی زد، برگ را برداشت، عطر خاک باران‌خورده را نفس کشید و با خودش گفت: «پاییز، زیباترین بهانه‌ی زمین برای دلتنگی است.»


In [9]:
# ==========================================
# The Final Combiner Chain (And the Missing Key Dilemma)
# ==========================================

final_prompt = ChatPromptTemplate.from_template(
    """
    موضوع: {topic}
    
    شعر عاشقانه:
    {poem}
    
    داستان:
    {story}
    
    وظیفه: با استفاده از شعر و توضیح بالا، یک نتیجه‌گیری فلسفی کوتاه بنویس.
    """
)

# Parallel Processing -> Final Prompt -> Model -> Text
final_chain = (
    map_chain 
    | final_prompt 
    | model 
    | StrOutputParser()
)

print("⏳ Attempting to execute the final chain...\n")

# This is expected to fail based on the lesson's explanation!
try:
    output = final_chain.invoke({"topic": "پاییز"})
    print(output)
except KeyError as e:
    print(f"❌ Expected Error: Missing Key {e}")
    print("\n💡 Lesson Note:")
    print("Yes! The map_chain only returned its own outputs ('poem' and 'story').")
    print("It threw away the original user input ('topic').")
    print("To pass the original input to the next step, we need 'RunnablePassthrough' (Next Lesson!).")

⏳ Attempting to execute the final chain...



❌ Expected Error: Missing Key "Input to ChatPromptTemplate is missing variables {'topic'}.  Expected: ['poem', 'story', 'topic'] Received: ['poem', 'story']\nNote: if you intended {topic} to be part of the string and not a variable, please escape it with double curly braces like: '{{topic}}'.\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/INVALID_PROMPT_INPUT "

💡 Lesson Note:
Yes! The map_chain only returned its own outputs ('poem' and 'story').
It threw away the original user input ('topic').
To pass the original input to the next step, we need 'RunnablePassthrough' (Next Lesson!).
